# Per-reason logistic regularization with fixed DBSCAN labels

User-selected broad grid: **C=0.1,1,10,100,500,1000,2000** for each of Safety, Funding, Efficacy, Recruitment, and Other. Smaller C means stronger L2 regularization. First vary one reason at a time while holding the other four at C=1000. This is 35 reason/C evaluations, representing **31 unique five-output configurations** because the five default points share one control.

Then shortlist each reason's strongest nondefault C **only if its one-at-a-time sample F1 ties or improves the original baseline**. Ties favor the C closest to 1000, then smaller C. Test every subset of two or more shortlisted changes together (at most 26 extra configurations), using the already fitted binary models. This is an adaptive combination check, not an exhaustive search over all 7^5 settings. Overall ties retain the original model, then fewer changes and settings closest to the original.

Keep the original 150 embeddings/scaler, DBSCAN generated targets and raw weights, normalized fitting-weight total 157 per reason, class weighting, L2 penalty, solver settings, cutoff 0.5, and highest-probability fallback fixed. Validation labels are used only to evaluate/select, never fit. No final refit with validation labels, Kaggle submission, commit, or tag. Preserve exact fixed data/normalized weights, individual fits, complete models, predictions, and settings.

In [1]:
from pathlib import Path
import hashlib
import itertools
import json
import platform
import warnings
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import f1_score
from threadpoolctl import threadpool_limits
from IPython.display import display

REASONS=["Safety","Funding","Efficacy","Recruitment","Other"]
FEATURES=[f"emb_{i}" for i in range(150)]
C_GRID=[0.1,1.0,10.0,100.0,500.0,1000.0,2000.0]
DEFAULT_C=1000.0
cwd=Path.cwd()
DATA=next(p for p in [cwd,cwd/"part1",cwd/"AML-Midterm"/"part1"]
          if (p/"part1_train.csv").is_file() or (p/"data/part1_train.csv").is_file()).resolve()
INPUT_DIR=DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
SOURCE=DATA/"experiments/additional_methods"
OUT=DATA/"experiments/per_reason_regularization"
for folder in ["datasets","reason_models","models","validation_predictions","submissions"]:(OUT/folder).mkdir(parents=True,exist_ok=True)
train=pd.read_csv(INPUT_DIR/"part1_train.csv");valid=pd.read_csv(INPUT_DIR/"part1_val.csv")
known=train[REASONS].isin([0,1]).all(axis=1).to_numpy()
original=train[REASONS].to_numpy(dtype=int);truth=valid[REASONS].to_numpy(dtype=int)
assert len(train)==1171 and known.sum()==157 and len(valid)==78
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isin(truth,[0,1]).all()
source_record=json.loads((SOURCE/"submission_manifest.json").read_text())
source_model_path=SOURCE/source_record["model_file"];source_dataset_path=SOURCE/"datasets/dbscan_m5_q25.csv"
source_model=joblib.load(source_model_path)
fixed=pd.read_csv(source_dataset_path,float_precision="round_trip")
assert fixed.ID.tolist()==train.ID.tolist()
Y=fixed[REASONS].to_numpy(dtype=int);W=fixed[[f"weight_{r}" for r in REASONS]].to_numpy()
np.testing.assert_array_equal(Y[known],original[known]);np.testing.assert_array_equal(W[known],np.ones((157,5)))
assert np.array_equal(W>0,Y!=-100)
(OUT/"datasets/fixed_source.csv").write_bytes(source_dataset_path.read_bytes())
scaler=source_model["scaler"];assert scaler.n_samples_seen_==157
XS=scaler.transform(train[FEATURES].to_numpy());VS=scaler.transform(valid[FEATURES].to_numpy())
NORMALIZED_W=W*157/W.sum(axis=0)
np.testing.assert_allclose(NORMALIZED_W.sum(axis=0),np.full(5,157),atol=1e-10)
weights_frame=train[["ID"]].copy()
for j,r in enumerate(REASONS):weights_frame[f"effective_weight_{r}"]=NORMALIZED_W[:,j]
weights_frame.to_csv(OUT/"datasets/normalized_fitting_weights.csv",index=False)
protected_paths=[INPUT_DIR/"part1_train.csv",INPUT_DIR/"part1_val.csv",source_model_path,source_dataset_path,
                 SOURCE/source_record["submission_file"],SOURCE/"labeling_models/dbscan_m5_q25.joblib",SOURCE/"submission_manifest.json"]
protected_hashes={str(p.relative_to(DATA)):hashlib.sha256(p.read_bytes()).hexdigest() for p in protected_paths}
print("C grid:",C_GRID,"157 verified training trials, fixed DBSCAN labels/weights, 78 validation trials.")


C grid: [0.1, 1.0, 10.0, 100.0, 500.0, 1000.0, 2000.0] 157 verified training trials, fixed DBSCAN labels/weights, 78 validation trials.


In [2]:
def predict_model(model,raw):
    scaled=model["scaler"].transform(np.asarray(raw))
    return np.column_stack([est.predict_proba(scaled)[:,list(est.classes_).index(1)] for est in model["estimators"]])


def label_predictions(probabilities):
    assert np.isfinite(probabilities).all() and ((probabilities>=0)&(probabilities<=1)).all()
    predicted=(probabilities>=0.5).astype(int);empty=predicted.sum(axis=1)==0
    rows=np.flatnonzero(empty);predicted[rows,probabilities[rows].argmax(axis=1)]=1
    return predicted,empty

reference_probability=predict_model(source_model,valid[FEATURES].to_numpy())
reference_prediction,_=label_predictions(reference_probability)
reference_sample=f1_score(truth,reference_prediction,average="samples",zero_division=0)
reference_macro=f1_score(truth,reference_prediction,average="macro",zero_division=0)
np.testing.assert_allclose(reference_sample,source_record["validation_sample_f1"],atol=1e-14)
CACHE={};PROB_CACHE={};FIT_ROWS=[]
with threadpool_limits(limits=1):
    for j,reason in enumerate(REASONS):
        included=W[:,j]>0
        for C in C_GRID:
            estimator=clone(source_model["estimators"][j]).set_params(C=C)
            assert estimator.class_weight is None and estimator.solver=="lbfgs" and estimator.l1_ratio==0.0
            with warnings.catch_warnings():
                warnings.simplefilter("error",ConvergenceWarning)
                estimator.fit(XS[included],Y[included,j],sample_weight=NORMALIZED_W[included,j])
            if C==DEFAULT_C:np.testing.assert_allclose(estimator.predict_proba(VS),source_model["estimators"][j].predict_proba(VS),rtol=1e-8,atol=1e-10)
            CACHE[(reason,C)]=estimator
            PROB_CACHE[(reason,C)]=estimator.predict_proba(VS)[:,list(estimator.classes_).index(1)]
            name=f"{reason}_C{C:g}".replace(".","p")
            joblib.dump(estimator,OUT/"reason_models"/f"{name}.joblib",compress=3)
            FIT_ROWS.append({"reason":reason,"C":C,"training_rows":int(included.sum()),"known_rows":int((included&known).sum()),
                             "generated_rows":int((included&~known).sum()),"fitting_weight_total":float(NORMALIZED_W[included,j].sum()),
                             "iterations":int(estimator.n_iter_.max()),"coefficient_norm":float(np.linalg.norm(estimator.coef_)),
                             "reason_model_file":f"reason_models/{name}.joblib"})
pd.DataFrame(FIT_ROWS).to_csv(OUT/"fit_audit.csv",index=False)
assert len(FIT_ROWS)==35
print("35 binary fits complete; all five C=1000 controls reproduce the original probabilities.")

CONFIGURATIONS={};SINGLE_ROWS=[]

def evaluate(C_by_reason,stage,name):
    key=tuple(float(C_by_reason[r]) for r in REASONS)
    if key in CONFIGURATIONS:return CONFIGURATIONS[key]
    estimators=[CACHE[(r,C_by_reason[r])] for r in REASONS]
    probability=np.column_stack([PROB_CACHE[(r,C_by_reason[r])] for r in REASONS])
    prediction,empty=label_predictions(probability)
    changed=[r for r in REASONS if C_by_reason[r]!=DEFAULT_C]
    sample=f1_score(truth,prediction,average="samples",zero_division=0)
    macro=f1_score(truth,prediction,average="macro",zero_division=0)
    model={**source_model,"estimators":estimators,"parameter":key[0] if len(set(key))==1 else None,
           "C_by_reason":dict(C_by_reason),"regularization_mode":"per_reason_L2","source_dataset":"datasets/fixed_source.csv",
           "normalized_weight_file":"datasets/normalized_fitting_weights.csv","prediction_cutoff":0.5,"nonempty_fallback":True}
    row={"experiment":name,"stage":stage,**{f"C_{r}":float(C_by_reason[r]) for r in REASONS},
         "changed_reasons":";".join(changed),"number_changed_reasons":len(changed),
         "distance_from_default":sum(abs(float(np.log10(C_by_reason[r]/DEFAULT_C))) for r in REASONS),
         "sample_f1":sample,"macro_f1":macro,
         **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,prediction,average=None,zero_division=0))),
         "delta_vs_current":sample-reference_sample,"fallback_trials":int(empty.sum()),
         "changed_trials_vs_current":int((prediction!=reference_prediction).any(axis=1).sum()),
         "is_reference":not changed,"kaggle_score":np.nan}
    frame=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
    for j,r in enumerate(REASONS):frame[f"probability_{r}"]=probability[:,j];frame[f"predicted_{r}"]=prediction[:,j]
    frame["used_fallback"]=empty
    frame["trial_f1"]=2*(truth*prediction).sum(axis=1)/(truth.sum(axis=1)+prediction.sum(axis=1))
    frame.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
    joblib.dump(model,OUT/"models"/f"{name}.joblib",compress=3)
    CONFIGURATIONS[key]=row
    pd.DataFrame(CONFIGURATIONS.values()).to_csv(OUT/"comparison.csv",index=False)
    return row

baseline_C={r:DEFAULT_C for r in REASONS}
evaluate(baseline_C,"reference","baseline_C1000")
for reason in REASONS:
    for C in C_GRID:
        config={**baseline_C,reason:C}
        name=f"solo_{reason}_C{C:g}".replace(".","p")
        result=evaluate(config,"one_reason",name)
        SINGLE_ROWS.append({"reason":reason,"C":C,"distance_from_default":abs(float(np.log10(C/DEFAULT_C))),
                            "is_default":C==DEFAULT_C,"experiment":result["experiment"],"sample_f1":result["sample_f1"],
                            "macro_f1":result["macro_f1"],"delta_vs_current":result["delta_vs_current"],
                            "reason_f1":result[f"f1_{reason}"],"changed_trials_vs_current":result["changed_trials_vs_current"]})
    subset=pd.DataFrame(SINGLE_ROWS).query("reason==@reason").sort_values(["sample_f1","is_default","distance_from_default","C"],ascending=[False,False,True,True],kind="stable")
    row=subset.iloc[0]
    print(f"{reason}: best standalone C={row.C:g}, combined sample F1={row.sample_f1:.6f}, macro F1={row.macro_f1:.6f}",flush=True)
single=pd.DataFrame(SINGLE_ROWS);single.to_csv(OUT/"individual_sweep.csv",index=False)
assert len(single)==35 and len(CONFIGURATIONS)==31
display(single.pivot(index="reason",columns="C",values="sample_f1").round(6))


35 binary fits complete; all five C=1000 controls reproduce the original probabilities.
Safety: best standalone C=1000, combined sample F1=0.910256, macro F1=0.811421


Funding: best standalone C=1000, combined sample F1=0.910256, macro F1=0.811421


Efficacy: best standalone C=1000, combined sample F1=0.910256, macro F1=0.811421


Recruitment: best standalone C=1000, combined sample F1=0.910256, macro F1=0.811421


Other: best standalone C=1000, combined sample F1=0.910256, macro F1=0.811421


C,0.1,1.0,10.0,100.0,500.0,1000.0,2000.0
reason,,,,,,,
Efficacy,0.901709,0.884615,0.871795,0.893162,0.910256,0.910256,0.897436
Funding,0.897436,0.905983,0.905983,0.910256,0.910256,0.910256,0.910256
Other,0.884615,0.888889,0.901709,0.905983,0.910256,0.910256,0.910256
Recruitment,0.897436,0.897436,0.897436,0.897436,0.910256,0.910256,0.897436
Safety,0.897436,0.901709,0.910256,0.910256,0.910256,0.910256,0.910256


In [3]:
best_by_reason=single.sort_values(["sample_f1","is_default","distance_from_default","C"],ascending=[False,False,True,True],kind="stable").groupby("reason",sort=False).head(1)
best_by_reason.to_csv(OUT/"best_standalone_by_reason.csv",index=False)
SHORTLIST={};SHORTLIST_ROWS=[]
for reason in REASONS:
    candidates=single[(single.reason==reason)&(~single.is_default)&(single.sample_f1>=reference_sample-1e-12)]
    if len(candidates):
        chosen=candidates.sort_values(["sample_f1","distance_from_default","C"],ascending=[False,True,True],kind="stable").iloc[0]
        SHORTLIST[reason]=float(chosen.C)
        SHORTLIST_ROWS.append({"reason":reason,"shortlisted_C":float(chosen.C),"standalone_sample_f1":float(chosen.sample_f1),
                               "standalone_macro_f1":float(chosen.macro_f1),"strictly_improves_baseline":bool(chosen.sample_f1>reference_sample+1e-12)})
shortlist_frame=pd.DataFrame(SHORTLIST_ROWS,columns=["reason","shortlisted_C","standalone_sample_f1","standalone_macro_f1","strictly_improves_baseline"])
shortlist_frame.to_csv(OUT/"combination_shortlist.csv",index=False)
print("Best nondefault settings that tie/improve baseline:",SHORTLIST)
COMBINATION_COUNT=0
for size in range(2,len(SHORTLIST)+1):
    for reasons in itertools.combinations(list(SHORTLIST),size):
        config={**baseline_C,**{r:SHORTLIST[r] for r in reasons}}
        name="combined_"+"_".join(f"{r}C{config[r]:g}".replace(".","p") for r in reasons)
        evaluate(config,"combined",name);COMBINATION_COUNT+=1
comparison=pd.DataFrame(CONFIGURATIONS.values())
assert len(comparison)==31+COMBINATION_COUNT and COMBINATION_COUNT<=26
ranked=comparison.sort_values(["sample_f1","is_reference","number_changed_reasons","distance_from_default","experiment"],ascending=[False,False,True,True,True],kind="stable")
best=ranked.iloc[0]
comparison.query("stage=='combined'").to_csv(OUT/"combination_results.csv",index=False)
print("Combined configurations:",COMBINATION_COUNT,"unique configurations total:",len(comparison))
print("Selected:",best.experiment,"sample F1:",best.sample_f1,"macro F1:",best.macro_f1)
display(ranked.head(12)[["experiment","sample_f1","macro_f1","delta_vs_current","changed_reasons","changed_trials_vs_current"]].round(6))


Best nondefault settings that tie/improve baseline: {'Safety': 500.0, 'Funding': 500.0, 'Efficacy': 500.0, 'Recruitment': 500.0, 'Other': 500.0}


Combined configurations: 26 unique configurations total: 57
Selected: baseline_C1000 sample F1: 0.9102564102564102 macro F1: 0.811421460284173


,experiment,sample_f1,macro_f1,delta_vs_current,changed_reasons,changed_trials_vs_current
0,baseline_C1000,0.910256,0.811421,0.0,,0
17,solo_Efficacy_C500,0.910256,0.811421,0.0,Efficacy,0
12,solo_Funding_C2000,0.910256,0.811421,0.0,Funding,0
11,solo_Funding_C500,0.910256,0.811421,0.0,Funding,0
30,solo_Other_C2000,0.910256,0.811421,0.0,Other,0
29,solo_Other_C500,0.910256,0.811421,0.0,Other,0
23,solo_Recruitment_C500,0.910256,0.811421,0.0,Recruitment,0
6,solo_Safety_C2000,0.910256,0.811421,0.0,Safety,0
5,solo_Safety_C500,0.910256,0.811421,0.0,Safety,0
10,solo_Funding_C100,0.910256,0.811421,0.0,Funding,0


In [4]:
selected=joblib.load(OUT/"models"/f"{best.experiment}.joblib")
selected_probability=predict_model(selected,valid[FEATURES].to_numpy())
for row in comparison.itertuples(index=False):
    model=joblib.load(OUT/"models"/f"{row.experiment}.joblib")
    for j,r in enumerate(REASONS):assert model["estimators"][j].C==getattr(row,f"C_{r}")
    frame=pd.read_csv(OUT/"validation_predictions"/f"{row.experiment}.csv",float_precision="round_trip")
    assert frame.ID.tolist()==valid.ID.tolist()
    probs=frame[[f"probability_{r}" for r in REASONS]].to_numpy();prediction=frame[[f"predicted_{r}" for r in REASONS]].to_numpy(dtype=int)
    np.testing.assert_allclose(predict_model(model,valid[FEATURES].to_numpy()),probs,rtol=1e-12,atol=1e-12)
    np.testing.assert_array_equal(label_predictions(probs)[0],prediction)
    np.testing.assert_allclose(f1_score(truth,prediction,average="samples",zero_division=0),row.sample_f1,atol=1e-14)
    np.testing.assert_allclose(f1_score(truth,prediction,average="macro",zero_division=0),row.macro_f1,atol=1e-14)
# Independently refit all five selected models from persisted fixed data and normalized weights.
frame=pd.read_csv(OUT/"datasets/fixed_source.csv",float_precision="round_trip")
weights=pd.read_csv(OUT/"datasets/normalized_fitting_weights.csv",float_precision="round_trip")
assert frame.ID.tolist()==train.ID.tolist()==weights.ID.tolist()
np.testing.assert_array_equal(frame[REASONS].to_numpy(dtype=int),Y)
np.testing.assert_array_equal(frame[[f"weight_{r}" for r in REASONS]].to_numpy(),W)
scaled=scaler.transform(frame[FEATURES].to_numpy());replay={**selected,"estimators":[]}
with threadpool_limits(limits=1):
    for j,r in enumerate(REASONS):
        w=weights[f"effective_weight_{r}"].to_numpy();included=w>0
        np.testing.assert_allclose(w,NORMALIZED_W[:,j],rtol=1e-12,atol=1e-12)
        estimator=clone(selected["estimators"][j])
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            estimator.fit(scaled[included],Y[included,j],sample_weight=w[included])
        replay["estimators"].append(estimator)
np.testing.assert_allclose(predict_model(replay,valid[FEATURES].to_numpy()),selected_probability,rtol=1e-8,atol=1e-10)
np.testing.assert_array_equal(label_predictions(predict_model(replay,valid[FEATURES].to_numpy()))[0],label_predictions(selected_probability)[0])
for rel,h in protected_hashes.items():assert hashlib.sha256((DATA/rel).read_bytes()).hexdigest()==h
print("All C=1000 controls, saved models/weights/scores, per-reason C settings, independent selected-model refit, and protected hashes passed.")


All C=1000 controls, saved models/weights/scores, per-reason C settings, independent selected-model refit, and protected hashes passed.


In [5]:
candidate=None
if best.sample_f1>reference_sample+1e-12:
    test=pd.read_csv(INPUT_DIR/"part1_test.csv");sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)
    assert set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
    prediction,empty=label_predictions(predict_model(selected,test[FEATURES].to_numpy()))
    output=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in prediction]})
    output=output.set_index("ID").loc[sample.ID].reset_index()
    assert output.shape==(312,2) and output.Predicted.str.len().gt(0).all()
    path=OUT/"submissions"/f"part1_dbscan_{best.experiment}_submission.csv"
    output.to_csv(path,index=False);pd.testing.assert_frame_equal(pd.read_csv(path),output)
    candidate={"experiment":best.experiment,"C_by_reason":selected["C_by_reason"],"validation_sample_f1":float(best.sample_f1),"validation_macro_f1":float(best.macro_f1),
               "submission_file":str(path.relative_to(OUT)),"submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
               "model_file":f"models/{best.experiment}.joblib","model_sha256":hashlib.sha256((OUT/"models"/f"{best.experiment}.joblib").read_bytes()).hexdigest(),
               "test_rows":312,"test_fallback_trials":int(empty.sum()),"kaggle_score":None,"kaggle_status":"prepared locally; not submitted"}
    (OUT/"submission_manifest.json").write_text(json.dumps(candidate,indent=2)+"\n")
settings={"C_grid":C_GRID,"default_C":DEFAULT_C,"reasons":REASONS,"binary_fits":35,"individual_reason_C_evaluations":35,
          "unique_one_reason_and_control_configurations":31,"combination_shortlist":SHORTLIST,"combination_configurations":COMBINATION_COUNT,
          "unique_total_configurations":len(comparison),"combination_rule":"one best nondefault C per reason if standalone sample F1 ties/improves baseline; all subsets of size >=2",
          "features":FEATURES,"classifier_scaler_fit_rows":157,"source_targets_changed":False,"source_raw_weights_changed":False,
          "normalized_weight_total_per_reason":157,"class_weight":None,"penalty":"L2","solver":"lbfgs","cutoff":0.5,"fallback":True,
          "validation_rows":78,"validation_used_for_fit":False,"final_validation_label_refit":False,
          "reference_sample_f1":reference_sample,"reference_macro_f1":reference_macro,"best_experiment":best.experiment,
          "best_sample_f1":float(best.sample_f1),"best_macro_f1":float(best.macro_f1),"best_C_by_reason":selected["C_by_reason"],
          "leader_changed":bool(best.sample_f1>reference_sample+1e-12),"candidate":candidate,"protected_input_sha256":protected_hashes,
          "dataset_file_sha256":{str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((OUT/"datasets").glob('*.csv'))},
          "selection":"sample F1; ties prefer baseline, fewer changed reasons, then C settings closest to default",
          "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"numpy":np.__version__,"pandas":pd.__version__},"new_kaggle_scores":None}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
lines=["# Per-reason logistic regularization sweep","",
       "User-selected C values: 0.1,1,10,100,500,1000,2000. Vary one classifier at a time with the other four at C=1000, "
       "evaluating the complete five-reason sample F1. This is 35 reason/C evaluations and 31 unique multi-output configurations. "
       "Every classifier uses the same 150 embeddings/scaler, fixed DBSCAN targets/raw weights, normalized weight total "
       "157 per reason, L2 penalty, class_weight=None, original solver settings, cutoff 0.5, and highest-probability fallback.","",
       f"Reference sample F1 **{reference_sample:.6f}**, macro F1 **{reference_macro:.6f}**. "
       f"Selected **{best.experiment}**: sample F1 **{best.sample_f1:.6f}**, macro F1 **{best.macro_f1:.6f}**. "
       f"Selected C by reason: `{selected['C_by_reason']}`.","",
       "## One-at-a-time sample F1","",
       "| Reason | "+" | ".join(f"C={C:g}" for C in C_GRID)+" |",
       "|---|"+"---:|"*len(C_GRID)]
for reason in REASONS:
    values=single.loc[single.reason==reason].set_index('C').sample_f1
    lines.append("| "+reason+" | "+" | ".join(f"{values.loc[C]:.6f}" for C in C_GRID)+" |")
lines.extend(["","## Adaptive combinations","",
       f"Shortlist: `{SHORTLIST}`. For each reason, select its strongest nondefault C only when the individual score "
       "ties/improves the original baseline; ties favor values closest to C=1000, then smaller C. "
       f"All subsets of at least two shortlisted changes were evaluated: **{COMBINATION_COUNT}** additional "
       f"configurations; **{len(comparison)}** unique configurations total. This is not an exhaustive 7^5 grid.","",
       "| Configuration | Sample F1 | Macro F1 | Change vs reference | Changed reasons |",
       "|---|---:|---:|---:|---|"])
for row in ranked.itertuples(index=False):
    lines.append(f"| {row.experiment} | {row.sample_f1:.6f} | {row.macro_f1:.6f} | {row.delta_vs_current:+.6f} | {row.changed_reasons or 'none'} |")
lines.extend(["","## Verification and artifacts","",
       "Each C=1000 reason fit reproduces its original probabilities. Every saved complete model reloads with matching "
       "probabilities, labels, and sample/macro F1. The selected model independently refits all five classifiers from the "
       "saved fixed data and normalized weight table with matching probabilities. Targets, raw weights, scaler, and source "
       "artifacts remain unchanged. Validation/test labels never enter fitting; no final validation-label refit was performed.","",
       "reason_models/ preserves all 35 binary fits. models/ and validation_predictions/ preserve every distinct complete "
       "configuration. datasets/ saves shared original data and exact normalized weights. fit_audit.csv records fitting "
       "rows/weights/convergence; individual_sweep.csv, best_standalone_by_reason.csv, combination_shortlist.csv, "
       "combination_results.csv, comparison.csv, and settings.json record settings, selections, results, and checksums.","",
       "The same repeatedly consulted 78 validation trials select settings, including the adaptive combination shortlist. "
       "These results do not establish independent test improvement. No Kaggle submission, commit, or tag was made.",""])
lines.append(f"New candidate prepared locally: `{candidate['submission_file']}`. Kaggle score unknown." if candidate else
             "No setting improves primary sample F1; retain the original all-C=1000 model and its existing prepared Kaggle file.")
(OUT/"SUMMARY.md").write_text("\n".join(lines)+"\n")
print("Selected C settings:",selected['C_by_reason'])
print("Leader changed:",settings['leader_changed'],"Candidate:",candidate)


Selected C settings: {'Safety': 1000.0, 'Funding': 1000.0, 'Efficacy': 1000.0, 'Recruitment': 1000.0, 'Other': 1000.0}
Leader changed: False Candidate: None
